# 03a — Silver Layer: Dimension Tables (dim_coin, dim_date, dim_date_hour)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# SILVER LAYER — DIMENSION TABLES
# New Data Model: Star Schema
#
#   dim_coin          — Master coin attributes (one row per coin)
#   dim_date          — Date dimension (calendar + trading day attributes)
#   dim_date_hour     — Hour-level dimension (for OHLC grain)
#
# ══════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType,
    BooleanType, DateType, TimestampType
)
from datetime import datetime, timezone, date, timedelta
 
DB_NAME = 'crypto_db'

# ── Dimension target tables ───────────────────────────────────────────────
DIM_COIN      = f'{DB_NAME}.dim_coin'
DIM_DATE      = f'{DB_NAME}.dim_date'
DIM_DATE_HOUR = f'{DB_NAME}.dim_date_hour'

# ── Source bronze tables (read only) ─────────────────────────────────────
BRONZE_MARKET  = f'{DB_NAME}.bronze_market_data'
BRONZE_OHLC    = f'{DB_NAME}.bronze_ohlc_data'
COIN_UNIVERSE  = f'{DB_NAME}.coin_universe'

def _now_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
 
def _today_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d')
 
print('Config loaded.')
for t in [DIM_COIN, DIM_DATE, DIM_DATE_HOUR]:
    print(f'  Target: {t}')

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# DIM 1 — dim_coin
# Grain : 1 row per coin_id (natural key = id from CoinGecko)
# Contains: static + slowly-changing coin metadata
# SCD Type: Type-1 (overwrite) — we always want the latest rank, name etc.
# ══════════════════════════════════════════════════════════════════════════


 
SCD2_TRACKED_COLS = [
    'market_cap_rank',
    'market_segment',
    'supply_utilisation_pct',
    'ath_distance_bucket',
    'is_scarce',
]
 
 
def build_dim_coin_source():
    """
    Build the incoming source DataFrame for dim_coin from bronze_market_data.
    Returns the pre-SCD-2 version of the coin dimension, ready for merge.
    """
    df = spark.table(BRONZE_MARKET)
    df = df.toDF(*[c.lower().strip().replace(' ', '_') for c in df.columns])
 
    # Only keep the latest snapshot per coin
    w_latest = Window.partitionBy('id').orderBy(F.col('ingestion_date').desc())
    df = (df
        .withColumn('_rn', F.row_number().over(w_latest))
        .filter(F.col('_rn') == 1)
        .drop('_rn')
    )
 
    # Cast supply columns
    for c in ['circulating_supply', 'total_supply', 'max_supply']:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(DoubleType()))
    if 'market_cap_rank' in df.columns:
        df = df.withColumn('market_cap_rank', F.col('market_cap_rank').cast(IntegerType()))
    if 'ath_change_percentage' in df.columns:
        df = df.withColumn('ath_change_percentage', F.col('ath_change_percentage').cast(DoubleType()))
 
    for col_name, dtype in df.dtypes:
        if dtype == 'string':
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
 


    # ── Market segment (derived, stored in dim for convenience) ───────────
    df = df.withColumn(
            'market_segment',
            F.when(F.col('market_cap_rank') <= 10,  'Top_10')
            .when(F.col('market_cap_rank') <= 50,  'Top_50')
            .when(F.col('market_cap_rank') <= 200, 'Top_200')
            .otherwise('Long_Tail')
        )

    # ── Supply utilisation ratio ──────────────────────────────────────────
    df = df.withColumn(
            'supply_utilisation_pct',
            F.when(
                F.col('max_supply').isNotNull() & (F.col('max_supply') > 0),
                F.round(F.col('circulating_supply') / F.col('max_supply') * 100, 2)
            )
        )

    # ── Scarcity flag (circulating > 90% of max) ─────────────────────────
    df = df.withColumn(
        'is_scarce',
        F.when(F.col('supply_utilisation_pct') >= 90, True).otherwise(False)
    )

    # ── ATH distance bucket ───────────────────────────────────────────────
    df = df.withColumn(
        'ath_distance_bucket',
        F.when(F.col('ath_change_percentage').isNull(), 'UNKNOWN')
         .when(F.col('ath_change_percentage') >= -5,    'AT_ATH')
         .when(F.col('ath_change_percentage') >= -20,   'NEAR_ATH')
         .when(F.col('ath_change_percentage') >= -50,   'MID_RANGE')
         .otherwise('DEEP_CORRECTION')
    )

     # ── Join confidentiality_score from coin_universe (graceful degradation) ─
    # These columns are expected from new bronze additions.
    # If coin_universe doesn't carry them yet, columns default to NULL.
    if spark.catalog.tableExists(COIN_UNIVERSE):
        universe_cols = spark.table(COIN_UNIVERSE).columns
        has_conf  = 'confidentiality_score'  in universe_cols
        has_tier  = 'data_transparency_tier' in universe_cols
 
        if has_conf or has_tier:
            sel = [F.col('coin_id').alias('id')]
            if has_conf:  sel.append('confidentiality_score')
            if has_tier:  sel.append('data_transparency_tier')
 
            universe_today = (spark.table(COIN_UNIVERSE)
                .filter(F.col('universe_date') == _today_utc())
                .select(*sel)
            )
            df = df.join(universe_today, on='id', how='left')
 
            if not has_conf:
                df = df.withColumn('confidentiality_score', F.lit(None).cast(IntegerType()))
            if not has_tier:
                df = df.withColumn('data_transparency_tier', F.lit(None).cast(StringType()))
        else:
            df = (df
                .withColumn('confidentiality_score',  F.lit(None).cast(IntegerType()))
                .withColumn('data_transparency_tier', F.lit(None).cast(StringType()))
            )
    else:
        df = (df
            .withColumn('confidentiality_score',  F.lit(None).cast(IntegerType()))
            .withColumn('data_transparency_tier', F.lit(None).cast(StringType()))
        )
 
    # effective_from = today's ingestion_date
    df = df.withColumn('effective_from', F.col('ingestion_date').cast(DateType()))
 
    return df

In [0]:
def build_dim_coin():
    """
    SCD Type-2 merge into dim_coin.
    Step 1: Expire rows where tracked cols have changed.
    Step 2: Insert new version rows for changed/new coins.
    """
    src_df = build_dim_coin_source()
    src_df.createOrReplaceTempView('_dim_coin_src')
    today_str = _today_utc()
 
    if not spark.catalog.tableExists(DIM_COIN) or 'is_current' not in spark.table(DIM_COIN).columns:
        # ── First run or schema migration: initialise with scd_version=1, is_current=true ────
        init_df = (src_df
            .withColumn('coin_sk',       F.monotonically_increasing_id())
            .withColumn('effective_to',  F.lit(None).cast(DateType()))
            .withColumn('is_current',    F.lit(True))
            .withColumn('scd_version',   F.lit(1))
        )
        init_df = init_df.select(
            'coin_sk', 'id',
            'symbol', 'name',
            'market_cap_rank', 'market_segment',
            'circulating_supply', 'total_supply', 'max_supply',
            'supply_utilisation_pct', 'is_scarce',
            'ath_distance_bucket',
            'confidentiality_score', 'data_transparency_tier',
            'effective_from', 'effective_to', 'is_current', 'scd_version',
        )
        (init_df.write.format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .saveAsTable(DIM_COIN))
        print(f'Created {DIM_COIN} (SCD Type-2 initialised)')
        return init_df
    # ── Step 1: Expire rows where tracked cols have changed ───────────────
    tracked_cond = ' OR '.join([
        f'tgt.{c} <> src.{c}'
        for c in SCD2_TRACKED_COLS
    ])
    spark.sql(f'''
        MERGE INTO {DIM_COIN} tgt
        USING _dim_coin_src src
        ON  tgt.id = src.id AND tgt.is_current = true
        WHEN MATCHED AND (
            {tracked_cond}
        ) THEN UPDATE SET
            tgt.is_current   = false,
            tgt.effective_to = src.effective_from
    ''')
    print(f'  Step 1: expired changed rows in {DIM_COIN}')
 
    # ── Step 2: Insert new version rows for changed/new coins ─────────────
    existing_current = (spark.table(DIM_COIN)
        .filter(F.col('is_current') == True)
        .select('id', F.col('scd_version').alias('max_version'))
    )
    src_enriched = (src_df
        .join(existing_current, on='id', how='left')
    )
    # New or changed = no current row exists
    new_rows = src_enriched.filter(F.col('max_version').isNull())
    if new_rows.count() > 0:
        # Determine scd_version for new rows
        max_versions = (spark.table(DIM_COIN)
            .groupBy('id')
            .agg(F.max('scd_version').alias('prev_max_version'))
        )
        new_rows = new_rows.join(max_versions, on='id', how='left')
        new_rows = (new_rows
            .withColumn('scd_version',
                F.coalesce(F.col('prev_max_version') + 1, F.lit(1)))
            .withColumn('coin_sk',      F.monotonically_increasing_id())
            .withColumn('effective_to', F.lit(None).cast(DateType()))
            .withColumn('is_current',   F.lit(True))
            .select(
                'coin_sk', 'id',
                'symbol', 'name',
                'market_cap_rank', 'market_segment',
                'circulating_supply', 'total_supply', 'max_supply',
                'supply_utilisation_pct', 'is_scarce',
                'ath_distance_bucket',
                'confidentiality_score', 'data_transparency_tier',
                'effective_from', 'effective_to', 'is_current', 'scd_version',
            )
        )
        (new_rows.write.format('delta')
            .mode('append')
            .option('mergeSchema', 'true')
            .saveAsTable(DIM_COIN))
        print(f'  Step 2: inserted {new_rows.count()} new version rows into {DIM_COIN}')
    else:
        print(f'  Step 2: no new version rows needed (no changes detected)')
 
    return spark.table(DIM_COIN).filter(F.col('is_current') == True)
 
df_dim_coin = build_dim_coin()
coin_cnt = spark.table(DIM_COIN).count()
print(f'dim_coin total rows (all versions): {coin_cnt}')
print(f'dim_coin current rows: {spark.table(DIM_COIN).filter(F.col("is_current")==True).count()}')
 
try:
    spark.sql(f'OPTIMIZE {DIM_COIN} ZORDER BY (market_cap_rank, effective_from)')
    print(f'✅ {DIM_COIN} ready — {coin_cnt} rows (all versions)')
except Exception as e:
    print(f'OPTIMIZE skipped: {e}')

In [0]:
# ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════
# DIM 2 — dim_date
# ══════════════════════════════════════════════════════════════════════════
 
def build_dim_date(start_date: date, end_date: date):
    delta  = (end_date - start_date).days + 1
    dates  = [start_date + timedelta(days=i) for i in range(delta)]
 
    rows = []
    for d in dates:
        iso_wd   = d.isoweekday()
        week_num = int(d.strftime('%W'))
        quarter  = (d.month - 1) // 3 + 1
        rows.append({
            'date_id':              int(d.strftime('%Y%m%d')),
            'date_actual':          d.isoformat(),
            'year':                 d.year,
            'quarter':              quarter,
            'quarter_label':        f'Q{quarter}',
            'month':                d.month,
            'month_name':           d.strftime('%B'),
            'month_abbr':           d.strftime('%b'),
            'week_of_year':         week_num,
            'day_of_year':          d.timetuple().tm_yday,
            'day_of_month':         d.day,
            'day_of_week':          iso_wd,
            'day_name':             d.strftime('%A'),
            'day_abbr':             d.strftime('%a'),
            'is_weekend':           iso_wd >= 6,
            'is_weekday':           iso_wd <= 5,
            'is_trading_day':       True,
            'year_month':           d.strftime('%Y-%m'),
            'year_quarter':         f'{d.year}-Q{quarter}',
            'year_week':            d.strftime('%Y-W%W'),
            'is_today':             d == date.today(),
            'is_past':              d < date.today(),
            'is_future':            d > date.today(),
        })
 
    schema = StructType([
        StructField('date_id',       IntegerType(),  False),
        StructField('date_actual',   StringType(),   False),
        StructField('year',          IntegerType(),  False),
        StructField('quarter',       IntegerType(),  False),
        StructField('quarter_label', StringType(),   False),
        StructField('month',         IntegerType(),  False),
        StructField('month_name',    StringType(),   False),
        StructField('month_abbr',    StringType(),   False),
        StructField('week_of_year',  IntegerType(),  False),
        StructField('day_of_year',   IntegerType(),  False),
        StructField('day_of_month',  IntegerType(),  False),
        StructField('day_of_week',   IntegerType(),  False),
        StructField('day_name',      StringType(),   False),
        StructField('day_abbr',      StringType(),   False),
        StructField('is_weekend',    BooleanType(),  False),
        StructField('is_weekday',    BooleanType(),  False),
        StructField('is_trading_day',BooleanType(),  False),
        StructField('year_month',    StringType(),   False),
        StructField('year_quarter',  StringType(),   False),
        StructField('year_week',     StringType(),   False),
        StructField('is_today',      BooleanType(),  False),
        StructField('is_past',       BooleanType(),  False),
        StructField('is_future',     BooleanType(),  False),
    ])
    return spark.createDataFrame(rows, schema=schema)
 
 
bm = spark.table(BRONZE_MARKET)
bm = bm.toDF(*[c.lower().strip() for c in bm.columns])
 
earliest_str = bm.agg(F.min('ingestion_date')).collect()[0][0]
earliest_dt  = datetime.strptime(str(earliest_str), '%Y-%m-%d').date()
end_dt       = date.today() + timedelta(days=90)
 
print(f'Date range: {earliest_dt} → {end_dt}  ({(end_dt - earliest_dt).days + 1} days)')
 
df_dim_date = build_dim_date(earliest_dt, end_dt)
 
(df_dim_date.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DIM_DATE))
 
spark.sql(f'OPTIMIZE {DIM_DATE} ZORDER BY (date_id)')
print(f'✅ {DIM_DATE} ready — {spark.table(DIM_DATE).count()} rows')
spark.table(DIM_DATE).orderBy('date_actual').display(5, truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════
# DIM 3 — dim_date_hour
# ══════════════════════════════════════════════════════════════════════════
 
def build_dim_date_hour(start_date: date, end_date: date):
    CANDLE_HOURS = {0, 4, 8, 12, 16, 20}
    delta = (end_date - start_date).days + 1
    rows  = []
 
    for day_offset in range(delta):
        d = start_date + timedelta(days=day_offset)
        for h in range(24):
            quarter = (d.month - 1) // 3 + 1
            rows.append({
                'date_hour_id':   int(d.strftime('%Y%m%d')) * 100 + h,
                'date_id':        int(d.strftime('%Y%m%d')),
                'date_actual':    d.isoformat(),
                'hour':           h,
                'hour_label':     f'{h:02d}:00',
                'candle_block':   (h // 4) * 4,
                'is_candle_open': h in CANDLE_HOURS,
                'trading_session':
                    'ASIA'    if 0  <= h < 8  else
                    'LONDON'  if 8  <= h < 13 else
                    'NY'      if 13 <= h < 21 else
                    'OFF',
                'year':           d.year,
                'month':          d.month,
                'day_of_week':    d.isoweekday(),
                'is_weekend':     d.isoweekday() >= 6,
            })
 
    schema = StructType([
        StructField('date_hour_id',    IntegerType(), False),
        StructField('date_id',         IntegerType(), False),
        StructField('date_actual',     StringType(),  False),
        StructField('hour',            IntegerType(), False),
        StructField('hour_label',      StringType(),  False),
        StructField('candle_block',    IntegerType(), False),
        StructField('is_candle_open',  BooleanType(), False),
        StructField('trading_session', StringType(),  False),
        StructField('year',            IntegerType(), False),
        StructField('month',           IntegerType(), False),
        StructField('day_of_week',     IntegerType(), False),
        StructField('is_weekend',      BooleanType(), False),
    ])
    return spark.createDataFrame(rows, schema=schema)
 
 
df_dim_date_hour = build_dim_date_hour(earliest_dt, end_dt)
 
(df_dim_date_hour.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DIM_DATE_HOUR))
 
spark.sql(f'OPTIMIZE {DIM_DATE_HOUR} ZORDER BY (date_id, hour)')
print(f'✅ {DIM_DATE_HOUR} ready — {spark.table(DIM_DATE_HOUR).count()} rows')


In [0]:
# ─────────────────────────────────────────────────────────
# ── Final verification ────────────────────────────────────────────────────
 
print('\n====== DIMENSION LAYER VERIFICATION ======')
 
print('\n--- dim_coin: current rows by rank ---')
spark.table(DIM_COIN).filter(F.col('is_current') == True).orderBy('market_cap_rank').select(
    'coin_sk', 'id', 'symbol', 'name', 'market_cap_rank', 'market_segment',
    'supply_utilisation_pct', 'ath_distance_bucket', 'is_scarce',
    'confidentiality_score', 'data_transparency_tier',
    'effective_from', 'effective_to', 'is_current', 'scd_version'
).display(5, truncate=False)
 
print('\n--- dim_coin: SCD-2 version history sample ---')
spark.table(DIM_COIN).orderBy('id', 'scd_version').display(10, truncate=False)
 
print('\n--- dim_date: sample rows ---')
spark.table(DIM_DATE).filter('is_today = true').display(1, truncate=False)
 
print('\n--- dim_date_hour: candle open hours only ---')
spark.table(DIM_DATE_HOUR).filter('is_candle_open = true').limit(6).display(truncate=False)
 
print('\n--- Row counts ---')
for t in [DIM_COIN, DIM_DATE, DIM_DATE_HOUR]:
    print(f'  {t}: {spark.table(t).count()}')
 
print('\n✅ Dimension layer complete.')